# Reproduce Figures 15-17: active learning without added response noise

This notebook reproduces Figures 15-17 from the active-learning section of the paper using the stored Excel results:

`2025-05-28-normal-noisy-test_functions_standarized_paper_parallel_length-results_z_noise_zero.xlsx`

The original simulation notebook can generate many noise scenarios, but the paper figures shown here use the clean case, `z_noise_factor = 0`. The synthetic variable `X3` is still present: it is a deliberately non-informative input used as a reference for detecting wasted exploration.

## Reference

**Main article:** Vallerio, M., del Rio Chanona, A., & Navarro-Brull, F. J. (2026). *All you need is noise - from feature selection to explainable industrial AI*. **Digital Chemical Engineering, 18**, 100290. https://doi.org/10.1016/j.dche.2025.100290

Main ideas used in this notebook:

- Active learning chooses the next experiment from the data already collected, instead of committing to a fixed design matrix.
- The acquisition function controls the balance between exploitation and exploration.
- A Gaussian-process model with automatic relevance determination (ARD) has one length scale per input. Large length scales indicate variables that the model sees as weakly informative.
- `X3` is a synthetic non-informative factor. If a real factor behaves like `X3`, it is a candidate to freeze or remove from the active-learning campaign.

In [ ]:
# Runtime setup. The notebook is intentionally light: it only needs pandas and Plotly.
import importlib.util
import subprocess
import sys
from pathlib import Path


def ensure(import_name, package_name=None):
    """Install a package only if it is missing in the current Python environment."""
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name or import_name])


ensure("openpyxl")
ensure("plotly")
ensure("kaleido")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Image, display

pd.set_option("display.max_columns", 50)

# Find the workbook whether Jupyter starts in the repo root or in 04_Act_Learning.
DATA_FILE = "2025-05-28-normal-noisy-test_functions_standarized_paper_parallel_length-results_z_noise_zero.xlsx"
for root in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    in_subdir = root / "04_Act_Learning" / DATA_FILE
    in_current = root / DATA_FILE
    if in_subdir.exists():
        REPO_ROOT = root
        DATA_PATH = in_subdir
        break
    if in_current.exists() and root.name == "04_Act_Learning":
        REPO_ROOT = root.parent
        DATA_PATH = in_current
        break
else:
    raise FileNotFoundError(f"Could not find {DATA_FILE}")

DATA_PATH

## Load the stored active-learning results

Each row is one experiment selected by the active-learning loop. The important columns are:

- `n_sim`: repeated simulation number.
- `run`: experiment number inside one simulation.
- `x1`, `x2`: the two real benchmark inputs.
- `x3_fake_var`: the deliberately non-informative input.
- `y_new`: clean benchmark response, because `z_noise_factor = 0` here.
- `x1_length_scale`, `x2_length_scale`, `x3_fake_var_length_scale`: ARD Gaussian-process length scales.
- `acq_func` and `xi`: acquisition function and exploration setting.

In [ ]:
raw = pd.read_excel(DATA_PATH)

# Use clearer labels in plots and keep only the clean Rosenbrock case used in the paper figures.
ACQ_LABELS = {
    "PI": "Probability of Improvement",
    "gp_hedge": "Gaussian Process Hedge",
    "EI": "Expected Improvement",
    "LCB": "Lower Confidence Bound",
}
ACQ_ORDER = ["PI", "gp_hedge", "EI", "LCB"]
ACQ_LABEL_ORDER = [ACQ_LABELS[a] for a in ACQ_ORDER]

paper = raw.loc[
    (raw["selected_benchmark"] == "rosenbrock")
    & (raw["z_noise_factor"] == 0)
    & (raw["low_length_scale_bound"] == 0.1)
    & (raw["high_length_scale_bound"] == 10)
].copy()

paper["acquisition function"] = pd.Categorical(
    paper["acq_func"].map(ACQ_LABELS),
    categories=ACQ_LABEL_ORDER,
    ordered=True,
)
paper["xi_label"] = np.where(paper["xi"].eq(0), "xi = 0 (exploitation)", "xi = 0.1 (mild exploration)")
paper["trajectory"] = paper["acq_func"].astype(str) + " | xi=" + paper["xi"].astype(str) + " | sim=" + paper["n_sim"].astype(str)

paper.shape, paper.head()

## Snippet from the original notebook: benchmark functions and no-noise response

The Excel file was created by the notebook already in this folder. The full simulation is expensive, so the reproducing notebook reads the saved results. The snippet below shows the key idea: calculate the clean benchmark response and optionally multiply by response noise. For the paper figures, `z_noise_factor = 0`, so no response noise is added.

In [ ]:
# Minimal version of the benchmark code used to generate the Excel file.
# It is included for transparency; the figures below read the already generated workbook.

def rosenbrock(x, y):
    """Rosenbrock benchmark transformed so larger values are better."""
    z = (1 - x) ** 2 + 100 * (y - x ** 2) ** 2
    z_scaled = np.log10(z / 100 + 1)
    return 1 / (z_scaled + 1)


def function_test(x, y, xvar_noise=0, z_noise_factor=0, z_noise_type="normal", benchmark_name="rosenbrock"):
    """Evaluate a benchmark response with optional multiplicative response noise."""
    _ = xvar_noise  # The third variable is intentionally non-informative for the response.

    if benchmark_name != "rosenbrock":
        raise ValueError("This compact example keeps only the Rosenbrock function used below.")

    f_val = rosenbrock(x, y)

    if z_noise_type == "normal":
        z_noise = np.random.normal(loc=0.0, scale=1.0, size=np.shape(f_val))
    elif z_noise_type == "uniform":
        z_noise = np.random.uniform(low=-0.5, high=0.5, size=np.shape(f_val))
    else:
        raise ValueError(f"Unknown noise type: {z_noise_type}")

    # In the paper workbook z_noise_factor is zero, so this returns the clean response.
    return f_val * (1 + z_noise_factor * z_noise)


function_test(1, 1, xvar_noise=2.5, z_noise_factor=0)

## Snippet from the original notebook: active-learning loop

The saved workbook was generated by repeatedly running a Gaussian-process optimizer over `X1`, `X2`, and the fake `X3` input. ARD length scales are recorded after every run. The following condensed version shows the structure without launching the full parallel job.

In [ ]:
# This cell is a readable template of the original simulation loop.
# Set RUN_EXPENSIVE_SIMULATION = True only if you want to regenerate results from scratch.
RUN_EXPENSIVE_SIMULATION = False

if RUN_EXPENSIVE_SIMULATION:
    import ProcessOptimizer as po
    from ProcessOptimizer.learning.gaussian_process.gpr import GaussianProcessRegressor
    from ProcessOptimizer.learning.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

    space = po.Space([
        po.Real(-2, 2, name="x1"),
        po.Real(-1, 3, name="x2"),
        po.Real(-3, 3, name="x3_fake_var"),  # Deliberately non-informative input.
    ])

    kernel = (
        ConstantKernel(1.0, constant_value_bounds=(1e-3, 1e3))
        * Matern(length_scale=[1.0, 1.0, 1.0], length_scale_bounds=(0.1, 10), nu=2.5)
        + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-10, 1e1))
    )

    gpr = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=10, random_state=0)
    opt = po.Optimizer(space, base_estimator=gpr, acq_func="PI", random_state=0, n_initial_points=9, lhs=True)
    opt.acq_func_kwargs = {"xi": 0.0}

    rows = []
    for run in range(1, 101):
        x1_new, x2_new, x3_new = opt.ask()
        y_new = function_test(x1_new, x2_new, x3_new, z_noise_factor=0, benchmark_name="rosenbrock")
        opt.tell([x1_new, x2_new, x3_new], -y_new)  # Optimizer minimizes, so maximize by using -y.

        if opt.models:
            length_scales = opt.models[-1].kernel_.k1.k2.length_scale
        else:
            length_scales = [np.nan, np.nan, np.nan]

        rows.append({
            "run": run,
            "x1": x1_new,
            "x2": x2_new,
            "x3_fake_var": x3_new,
            "y_new": y_new,
            "x1_length_scale": length_scales[0],
            "x2_length_scale": length_scales[1],
            "x3_fake_var_length_scale": length_scales[2],
        })

## Figure 15: active-learning trajectories

This figure shows where the optimizer samples in the real `X1-X2` plane. Each thin path is one of the 50 repeated simulations. Markers are colored by run number, from early experiments to late experiments, using the same progressive idea as the paper figure. The star marks the standardized Rosenbrock optimum at `X1 = 0`, `X2 = 0`.

In [ ]:
fig15_df = paper.loc[paper["xi"].isin([0, 0.1])].copy()
fig15_df["xi_label"] = pd.Categorical(
    fig15_df["xi_label"],
    categories=["xi = 0 (exploitation)", "xi = 0.1 (mild exploration)"],
    ordered=True,
)

fig15 = make_subplots(
    rows=2,
    cols=4,
    shared_xaxes=True,
    shared_yaxes=True,
    horizontal_spacing=0.025,
    vertical_spacing=0.11,
    subplot_titles=ACQ_LABEL_ORDER,
    row_titles=["xi = 0", "xi = 0.1"],
)

# Add one trajectory per simulation. Lines show the path; markers carry the run color.
for row_i, xi_value in enumerate([0.0, 0.1], start=1):
    for col_i, acq in enumerate(ACQ_ORDER, start=1):
        panel = fig15_df.loc[(fig15_df["xi"] == xi_value) & (fig15_df["acq_func"] == acq)]
        first_panel_sim = int(panel["n_sim"].min())
        for sim, sim_df in panel.groupby("n_sim"):
            sim_df = sim_df.sort_values("run")
            show_run_scale = bool(row_i == 1 and col_i == 4 and int(sim) == first_panel_sim)
            fig15.add_trace(
                go.Scatter(
                    x=sim_df["x2"],
                    y=sim_df["x1"],
                    mode="lines+markers",
                    line=dict(color="rgba(15, 23, 42, 0.28)", width=0.7),
                    marker=dict(
                        size=4,
                        color=sim_df["run"],
                        colorscale="Magma",
                        cmin=1,
                        cmax=100,
                        opacity=0.88,
                        colorbar=dict(title="Run", len=0.50, x=1.015, y=0.72) if show_run_scale else None,
                        showscale=show_run_scale,
                    ),
                    customdata=np.stack([sim_df["n_sim"], sim_df["run"], sim_df["y_new"]], axis=-1),
                    hovertemplate="Simulation %{customdata[0]}<br>Run %{customdata[1]}<br>X2=%{x:.3f}<br>X1=%{y:.3f}<br>Y=%{customdata[2]:.3f}<extra></extra>",
                    showlegend=False,
                ),
                row=row_i,
                col=col_i,
            )

        fig15.add_trace(
            go.Scatter(
                x=[0],
                y=[0],
                mode="markers",
                marker=dict(symbol="star", size=15, color="white", line=dict(color="black", width=1.5)),
                name="Global optimum",
                hovertemplate="Global optimum<br>X2=0<br>X1=0<extra></extra>",
                showlegend=(row_i == 1 and col_i == 1),
            ),
            row=row_i,
            col=col_i,
        )

fig15.update_xaxes(title_text="X2", range=[-1.1, 3.1], dtick=1, showgrid=True, zeroline=True, zerolinecolor="rgba(0,0,0,0.35)", zerolinewidth=1)
fig15.update_yaxes(title_text="X1", range=[-2.1, 2.1], dtick=1, showgrid=True, zeroline=True, zerolinecolor="rgba(0,0,0,0.35)", zerolinewidth=1)
fig15.update_layout(
    title="Figure 15. Active-learning trajectories for the clean Rosenbrock example",
    height=620,
    width=1180,
    template="plotly_white",
    margin=dict(l=70, r=180, t=95, b=65),
    legend=dict(orientation="v", yanchor="top", y=0.42, xanchor="left", x=1.015),
)
display(Image(fig15.to_image(format="png", width=1180, height=620, scale=2)))

**Fig. 15.** Active-learning trajectories are highly sensitive to the choice of acquisition function and exploration bias. Search paths are shown for four common acquisition functions--Probability of Improvement (PI), Expected Improvement (EI), GP-Hedge, and Lower Confidence Bound (LCB)--under two exploration settings: ξ = 0 (top, pure exploitation) and ξ = 0.1 (bottom, mild exploration).

## Figure 16: length scales as an early diagnostic

For Figure 16 we focus on the Probability of Improvement case during the first 30 runs. The layout follows the paper: response `Y` on top, `X1` and `X2` together with a dashed optimum line at zero, the non-informative `X3` alone, and then the three Gaussian-process length scales on a log scale. The solid lines are medians over 50 simulations and the shaded bands show the interquartile range.

In [ ]:
def summarize_for_lanes(df, value_columns, group_columns=("run",)):
    """Return median and interquartile range for the selected columns."""
    long = df.melt(
        id_vars=list(group_columns),
        value_vars=list(value_columns),
        var_name="series",
        value_name="value",
    )
    summary = (
        long.groupby([*group_columns, "series"], observed=True)["value"]
        .quantile([0.25, 0.5, 0.75])
        .unstack()
        .reset_index()
        .rename(columns={0.25: "q25", 0.5: "median", 0.75: "q75"})
    )
    return summary


COLORS = {
    "Y": "#b832ff",
    "X1": "#111827",
    "X2": "#2563eb",
    "X3 (non-informative)": "#b91c1c",
    "X1 length scale": "#111827",
    "X2 length scale": "#2563eb",
    "X3 length scale": "#b91c1c",
}
SERIES_LABELS = {
    "y_new": "Y",
    "x1": "X1",
    "x2": "X2",
    "x3_fake_var": "X3 (non-informative)",
    "x1_length_scale": "X1 length scale",
    "x2_length_scale": "X2 length scale",
    "x3_fake_var_length_scale": "X3 length scale",
}


def add_iqr_trace(fig, data, label, row, col=1, showlegend=True):
    """Add an interquartile band plus a median line to a subplot lane."""
    d = data.loc[data["label"] == label].sort_values("run")
    color = COLORS[label]
    rgba = {
        "#b832ff": "rgba(184, 50, 255, 0.16)",
        "#111827": "rgba(17, 24, 39, 0.16)",
        "#2563eb": "rgba(37, 99, 235, 0.16)",
        "#b91c1c": "rgba(185, 28, 28, 0.16)",
    }[color]

    fig.add_trace(
        go.Scatter(
            x=pd.concat([d["run"], d["run"].iloc[::-1]]),
            y=pd.concat([d["q75"], d["q25"].iloc[::-1]]),
            fill="toself",
            fillcolor=rgba,
            line=dict(color="rgba(255,255,255,0)"),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=row,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=d["run"],
            y=d["median"],
            mode="lines",
            name=label,
            legendgroup=label,
            showlegend=showlegend,
            line=dict(color=color, width=2),
            hovertemplate=f"{label}<br>Run=%{{x}}<br>Median=%{{y:.3g}}<extra></extra>",
        ),
        row=row,
        col=col,
    )


def build_lane_figure(summary, title, facet_value=None, show_column_titles=False):
    """Build the paper-style four-lane active-learning diagnostic plot."""
    if facet_value is None:
        fig = make_subplots(
            rows=4,
            cols=1,
            shared_xaxes=True,
            vertical_spacing=0.035,
            row_heights=[0.22, 0.28, 0.24, 0.26],
        )
        cols = [(1, summary)]
    else:
        fig = make_subplots(
            rows=4,
            cols=len(ACQ_LABEL_ORDER),
            shared_xaxes=True,
            shared_yaxes="rows",
            vertical_spacing=0.04,
            horizontal_spacing=0.025,
            row_heights=[0.22, 0.28, 0.24, 0.26],
            subplot_titles=ACQ_LABEL_ORDER if show_column_titles else None,
        )
        cols = []
        for i, label in enumerate(ACQ_LABEL_ORDER, start=1):
            cols.append((i, summary.loc[summary[facet_value] == label]))

    for col_i, col_summary in cols:
        for label in ["Y"]:
            add_iqr_trace(fig, col_summary, label, row=1, col=col_i, showlegend=(col_i == 1))
        for label in ["X1", "X2"]:
            add_iqr_trace(fig, col_summary, label, row=2, col=col_i, showlegend=(col_i == 1))
        fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.65, row=2, col=col_i)
        for label in ["X3 (non-informative)"]:
            add_iqr_trace(fig, col_summary, label, row=3, col=col_i, showlegend=(col_i == 1))
        for label in ["X1 length scale", "X2 length scale", "X3 length scale"]:
            add_iqr_trace(fig, col_summary, label, row=4, col=col_i, showlegend=(col_i == 1))

        fig.update_yaxes(title_text="Y", row=1, col=col_i, range=[0.35, 1.03])
        fig.update_yaxes(title_text="X1 & X2", row=2, col=col_i, range=[-3.25, 3.25])
        fig.update_yaxes(title_text="X3", row=3, col=col_i, range=[-3.2, 3.2])
        fig.update_yaxes(title_text="Length scales", row=4, col=col_i, type="log", range=[-1.2, 1.15])
        fig.update_xaxes(title_text="Run", row=4, col=col_i)

    fig.update_layout(
        title_text=title,
        template="plotly_white",
        height=780 if facet_value is None else 820,
        width=920 if facet_value is None else 1280,
        margin=dict(l=80, r=185, t=115, b=65),
        legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02),
        title=dict(y=0.985, x=0.5, xanchor="center", yanchor="top"),
    )
    return fig


fig16_source = paper.loc[(paper["acq_func"] == "PI") & (paper["xi"] == 0) & (paper["run"] <= 30)].copy()
fig16_summary = summarize_for_lanes(fig16_source, SERIES_LABELS.keys())
fig16_summary["label"] = fig16_summary["series"].map(SERIES_LABELS)

fig16 = build_lane_figure(
    fig16_summary,
    "Figure 16. Early length-scale diagnostic for the Probability of Improvement case",
)
fig16.add_vrect(x0=1, x1=9, fillcolor="lightgray", opacity=0.18, line_width=0, row="all", col=1)
fig16.add_vrect(x0=9, x1=30, fillcolor="khaki", opacity=0.16, line_width=0, row="all", col=1)
fig16.add_annotation(x=5, y=0.99, text="Initial design", showarrow=False, row=1, col=1)
fig16.add_annotation(x=20, y=0.99, text="Active learning", showarrow=False, row=1, col=1)
display(Image(fig16.to_image(format="png", width=920, height=780, scale=2)))

**Fig. 16.** Gaussian-process length scales provide an early diagnostic of non-informative inputs during active learning. Following the initial exploration phase, the length scale associated with the dummy variable X3 rapidly saturates at the upper prior bound, indicating practical irrelevance, while the informative variables X1 and X2 remain well constrained. Solid lines show the median trajectory and shaded bands the interquartile range, aggregated over 50 independent simulations.

## Figure 17: what happens if the diagnostic is ignored

The same four-lane layout is repeated for all 100 runs and all four acquisition functions under pure exploitation (`xi = 0`). `X3` keeps moving even though its length scale indicates that it is not useful for the response.

In [ ]:
fig17_source = paper.loc[paper["xi"] == 0].copy()
fig17_summary = summarize_for_lanes(
    fig17_source,
    SERIES_LABELS.keys(),
    group_columns=("run", "acquisition function"),
)
fig17_summary["label"] = fig17_summary["series"].map(SERIES_LABELS)

fig17 = build_lane_figure(
    fig17_summary,
    "Figure 17. Continued perturbation of a non-informative input when diagnostics are ignored",
    facet_value="acquisition function",
    show_column_titles=True,
)
display(Image(fig17.to_image(format="png", width=1280, height=820, scale=2)))

**Fig. 17.** If length-scale diagnostics are ignored, active learning continues to explore non-informative variables despite no additional information gain. As shown in the preceding figure, the Gaussian-process length scale associated with the dummy factor X3 rapidly saturates at the upper prior bound after ~20 runs, indicating practical irrelevance. Nevertheless, most acquisition functions continue to perturb this dimension after 30 runs, even under pure exploitation and in the absence of measurement noise. Solid lines show the median trajectory across repeated simulations; shaded bands indicate variability (interquartile range).

## Practical takeaway

The clean response case is useful because it removes measurement noise as an explanation. Even with perfect responses, the optimizer may continue changing a non-informative factor. For process development, the practical rule is simple: keep one known non-informative reference input, monitor the ARD length scales, and freeze variables whose length scales behave like the reference.